# Data Monitoring — Metrics Analysis

Visualize observation data exported from the data-monitoring-service.

## Setup

Export the metrics from the SPARQL endpoint using the query below, save as `metrics.csv`:

```sparql
PREFIX mon: <http://lblod.data.gift/vocabularies/monitoring/>
PREFIX dct: <http://purl.org/dc/terms/>

SELECT ?queryName ?metricName ?value ?created
WHERE {
  ?obs a mon:Observation ;
    mon:forQuery ?query ;
    mon:metricName ?metricName ;
    mon:value ?value ;
    mon:partOfRun ?run .
  ?run dct:created ?created .
  BIND(REPLACE(STR(?query), "^.*/", "") AS ?queryName)
}
ORDER BY ?queryName DESC(?created) ?metricName
```

Expected CSV columns: `queryName`, `metricName`, `value`, `created`

In [ ]:
%pip install -q pandas matplotlib

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

CSV_PATH = "metrics.csv"

df = pd.read_csv(CSV_PATH)
df["created"] = pd.to_datetime(df["created"])
df["value"] = pd.to_numeric(df["value"])

# Observations within the same batch run are written seconds apart.
# Bucket into actual runs by rounding to 5-minute windows.
df["run"] = df["created"].dt.floor("5min")

df.sort_values("run", inplace=True)

print(f"{len(df)} observations, {df['queryName'].nunique()} queries, {df['run'].nunique()} runs")
df.head()

## Latest snapshot

Most recent values per query and metric.

In [ ]:
latest = df.loc[df.groupby(["queryName", "metricName"])["run"].idxmax()]

# Pivot: rows = queries, columns = metrics
pivot = latest.pivot(index="queryName", columns="metricName", values="value")

# Reorder so totalRows comes first
cols = ["totalRows"] + sorted(c for c in pivot.columns if c != "totalRows")
pivot = pivot[[c for c in cols if c in pivot.columns]]

pivot

## Coverage heatmaps

Per-query heatmap of optional field coverage (%) across runs.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

cmap = LinearSegmentedColormap.from_list("coverage", ["#d73027", "#fee08b", "#1a9850"])

comp = df[df["metricName"].str.startswith("completeness_")].copy()
totals = df[df["metricName"] == "totalRows"].set_index(["queryName", "run"])["value"].rename("total")

if len(comp):
    comp = comp.join(totals, on=["queryName", "run"])
    comp["pct"] = (comp["value"] / comp["total"] * 100).round(1)
    comp["field"] = comp["metricName"].str.replace("completeness_", "", regex=False)
    comp["run_label"] = comp["run"].dt.strftime("%Y-%m-%d %H:%M")

    queries = sorted(comp["queryName"].unique())
    for query in queries:
        qdf = comp[comp["queryName"] == query]
        heat = qdf.pivot_table(index="field", columns="run_label", values="pct", sort=False)
        heat = heat.sort_index()
        if heat.empty:
            continue

        fig, ax = plt.subplots(figsize=(max(4, len(heat.columns) * 1.6 + 1), max(1.5, len(heat) * 0.5 + 1)))
        im = ax.imshow(heat.values, aspect="auto", cmap=cmap, vmin=0, vmax=100)
        ax.set_xticks(range(len(heat.columns)))
        ax.set_xticklabels(heat.columns, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(len(heat.index)))
        ax.set_yticklabels(heat.index, fontsize=8)
        for i in range(len(heat)):
            for j in range(len(heat.columns)):
                v = heat.iloc[i, j]
                if pd.notna(v):
                    ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7,
                            color="white" if v < 40 else "black")
        ax.set_title(query, fontsize=10)
        fig.colorbar(im, ax=ax, label="%", shrink=0.8)
        plt.tight_layout()
        plt.show()
else:
    print("No completeness metrics found")

## Row count over time

Track `totalRows` per query across runs to spot sudden drops or spikes.

In [ ]:
totals = df[df["metricName"] == "totalRows"]

if len(totals) and totals["run"].nunique() > 1:
    fig, ax = plt.subplots(figsize=(14, 6))
    for name, group in totals.groupby("queryName"):
        ax.plot(group["run"], group["value"], marker="o", markersize=3, label=name)
    ax.set_ylabel("totalRows")
    ax.set_title("Row count over time")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Need at least 2 runs to plot trends")

## Change detection

Percentage change between the latest and previous run per metric. Highlights potential anomalies.

In [ ]:
runs = sorted(df["run"].unique())
if len(runs) >= 2:
    prev = df[df["run"] == runs[-2]].set_index(["queryName", "metricName"])["value"]
    curr = df[df["run"] == runs[-1]].set_index(["queryName", "metricName"])["value"]
    combined = pd.DataFrame({"previous": prev, "current": curr}).dropna()
    combined["change_%"] = ((combined["current"] - combined["previous"]) / combined["previous"] * 100).round(2)
    combined = combined.sort_values("change_%")

    flagged = combined[combined["change_%"].abs() > 10]
    if len(flagged):
        print(f"{len(flagged)} metrics changed by more than 10%:\n")
        display(flagged)
    else:
        print("No significant changes between the last two runs")
else:
    print("Need at least 2 runs to compare")